# edge-adapt: Colab에서 한 번에 (데이터 준비 → 모델 학습 → 본 실험)
노트북에 GPU가 없을 때 **오프라인 작업만** Colab GPU에서 한다. 실시간 데모는 노트북에서 한다 (GUIDE.md 8.3절).

**실행 순서**: 위에서 아래로 셀의 ▶ 버튼(또는 Shift+Enter)을 하나씩 누른다. 각 셀 설명의 **확인할 것**을 보고 다음으로 넘어간다.
**미리 할 일**: 구글 드라이브(https://drive.google.com)의 **내 드라이브**에 `edge-adapt` 폴더를 만들고, 받은 `edge-adapt.zip`을 그 안에 올린다.
**끊겼을 때**: Colab 무료 세션은 끊길 수 있다. 다시 연결하면 **셀 2, 3부터 다시 실행**하면 된다. 끝난 일(데이터 준비, 학습, 실험 결과)은 드라이브에 저장돼 있어 건너뛴다.
**이 노트북은 작성 환경에서 실행해 보지 못했다.** 셀이 실패하면 오류 메시지 전체를 팀에 공유한다.

**무료 Colab 전제** (정책은 Google 이 바꿀 수 있고 한도는 공개되지 않아 시간은 단정하지 않는다)
- 무료 GPU 는 보장되지 않는다: 할당이 안 되거나, 사용량이 많으면 몇 시간 뒤 끊기거나, 하루 이상 막힐 수 있다.
- 실행 중에는 **브라우저 탭을 닫지 않는다.** 노트북 절전·네트워크 끊김도 세션 종료 원인이므로 전원을 연결하고 절전을 끈다.
- 메모리(RAM)는 약 12GB, 드라이브 무료 용량은 15GB 안팎이라 **큰 파일은 드라이브에 쌓지 않는다.** 이 노트북은 `data_prepared`(640x400 JPEG)·모델·결과만 저장한다.
- 그래서 기본값을 **`PROFILE = "free"`** (아래 '실행 규모 선택' 셀)로 줄여 두었다: 시퀀스 수와 epoch 를 계획서 값보다 줄인 것이다. 계획서 값으로 돌리려면 `PROFILE = "full"` 로 바꾸고 드라이브 연결 셀부터 다시 실행한다. **줄인 값으로 얻은 결과는 보고서에 '무료 Colab 한도 때문에 축소'라고 적어야 한다.**
- 한 번에 끝내려 하지 말고, 학습 1 epoch 의 소요 시간(`yolo` 출력)에 남은 epoch 수를 곱해 한 번의 세션에 들어가는지 판단한다. 안 들어가면 같은 셀을 다시 실행해 이어서 학습한다 (자동).

## 1. GPU 확인
메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU → 저장** 을 먼저 한 뒤 실행한다. **확인할 것**: 표에 `Tesla T4` 같은 GPU 이름이 보인다. 'command not found'면 위 설정이 안 된 것이다.

In [ ]:
!nvidia-smi | head -12

## 2. 드라이브 연결, 저장소 풀기
실행하면 **'Google 드라이브 연결 허용'** 창이 뜬다 → **드라이브에 연결** → 계정 선택 → **허용**. **확인할 것**: `OK: /content/edge-adapt` 가 출력된다.

In [ ]:
import os, shutil, subprocess
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/edge-adapt'
assert os.path.exists(f'{D}/edge-adapt.zip'), f'{D}/edge-adapt.zip 이 없습니다. 드라이브의 내 드라이브/edge-adapt 폴더에 zip 을 올리세요.'
if not os.path.exists('/content/edge-adapt/src'):
    subprocess.run(['unzip', '-q', '-o', f'{D}/edge-adapt.zip', '-d', '/content'], check=True)
os.chdir('/content/edge-adapt')
for n in ['results', 'models', 'runs']:                      # 결과·모델·학습 기록은 드라이브에 저장 (세션이 끊겨도 남음)
    os.makedirs(f'{D}/{n}', exist_ok=True)
    if os.path.islink(n): os.remove(n)
    elif os.path.exists(n): shutil.rmtree(n)
    os.symlink(f'{D}/{n}', n)
print('OK:', os.getcwd())

## 3. 실행 규모 선택 (PROFILE)
`free` = 무료 Colab 한도에 맞춘 축소값 / `full` = 계획서 값. 기본은 `free`. **확인할 것**: 선택된 값이 출력된다. 값을 바꾸면 드라이브 연결 셀부터 다시 실행한다.

In [ ]:
PROFILE = "free"      # "free" 또는 "full"
P = {"free": dict(n_student=200, n_teacher=450, student_epochs=30, teacher_epochs=15, budgets="16", seeds="0", tune="--quick"),
     "full": dict(n_student=300, n_teacher=750, student_epochs=50, teacher_epochs=30, budgets="8 16 32", seeds="0 1 2", tune="")}[PROFILE]
with open('/content/edge-adapt/profile.env', 'w') as f:
    f.write(f'PROFILE={PROFILE}\nN_STUDENT={P["n_student"]}\nN_TEACHER={P["n_teacher"]}\nSTUDENT_EPOCHS={P["student_epochs"]}\nTEACHER_EPOCHS={P["teacher_epochs"]}\nBUDGETS="{P["budgets"]}"\nSEEDS="{P["seeds"]}"\nTUNE_FLAGS="{P["tune"]}"\n')
print(PROFILE, P)

## 4. 패키지 설치 (1회)
**확인할 것**: 오류 없이 끝난다 (노란 경고는 무시). 끝나면 메뉴 **런타임 → 세션 다시 시작**을 누르고, **셀 2, 3을 다시 실행한 뒤 이 셀은 건너뛰고** 셀 5로 간다. (numpy 버전 고정 때문에 재시작이 필요하다.)

In [ ]:
!pip install -q -r requirements-server.txt

## 5. SHIFT 데이터 크기 확인 (다운로드 전)
**확인할 것**: `img.tar` 크기(GB)가 출력되고, 아래 `df` 의 남은 용량이 그 값의 **2배 이상**이다. '미러에 … 가 없습니다'가 나오면 멈추고 팀에 공유한다 (GUIDE 4.1절).

In [ ]:
%%bash
cd /content/edge-adapt
python scripts/download_shift_hf.py --split val --dry-run || true
python scripts/download_shift_hf.py --split train --dry-run || true
df -h /content | tail -1

## 6. 데이터 준비 (다운로드 → 시퀀스 선택 → 프레임 추출 → YOLO 변환)
시간이 오래 걸린다. 이미 한 번 끝났다면 드라이브의 `data_prepared.zip`을 풀고 건너뛴다.
**확인할 것**: `select_seqs` 가 '열을 찾지 못함'으로 멈추면 GUIDE 4.3절대로 `--inspect` 출력의 열 이름을 확인한다. 끝에 `data_prepared.zip 저장 완료`가 나온다.

In [ ]:
%%bash
set -e
cd /content/edge-adapt
source profile.env
D=/content/drive/MyDrive/edge-adapt
Z=$D/data_prepared_$PROFILE.zip
if [ -f $Z ]; then unzip -q -o $Z -d /content/edge-adapt; echo "복원 완료"; exit 0; fi
python scripts/download_shift_hf.py --split val
python scripts/download_shift_hf.py --split train
V=data/raw/discrete/videos
python scripts/select_seqs.py --seq-csv $V/val/front/seq.csv --inspect
python scripts/select_seqs.py --train-csv $V/train/front/seq.csv --val-csv $V/val/front/seq.csv --seed 0 --n-student $N_STUDENT --n-teacher $N_TEACHER --seg-frames 500
python scripts/extract_seqs.py --tar $V/train/front/img.tar --seqs data/lists/student_train.txt data/lists/teacher_train.txt --out data/raw/img_train --every 10
python scripts/extract_seqs.py --tar $V/train/front/img.tar --seqs data/lists/stream_train.txt --out data/raw/img_train --every 1
python scripts/extract_seqs.py --tar $V/val/front/img.tar --seqs data/lists/student_val.txt data/lists/teacher_val.txt --out data/raw/img_val --every 10
python scripts/extract_seqs.py --tar $V/val/front/img.tar --seqs data/lists/stream_eval.txt --out data/raw/img_val --every 1
Y="python scripts/shift_to_yolo.py"; TJ=$V/train/front/det_2d.json; VJ=$V/val/front/det_2d.json
$Y --det2d $TJ --images data/raw/img_train --seqs data/lists/student_train.txt --out data/yolo/student_train --step 10 --drop-empty
$Y --det2d $TJ --images data/raw/img_train --seqs data/lists/teacher_train.txt --out data/yolo/teacher_train --step 10 --drop-empty
$Y --det2d $VJ --images data/raw/img_val --seqs data/lists/student_val.txt --out data/yolo/student_val --step 10 --drop-empty
$Y --det2d $VJ --images data/raw/img_val --seqs data/lists/teacher_val.txt --out data/yolo/teacher_val --step 10 --drop-empty
$Y --det2d $TJ --images data/raw/img_train --seqs data/lists/stream_train.txt --out data/yolo/stream_train --step 1
$Y --det2d $VJ --images data/raw/img_val --seqs data/lists/stream_eval.txt --out data/yolo/stream_eval --step 1
python scripts/vis_check.py data/yolo/student_train --n 8
python scripts/vis_check.py data/yolo/stream_eval --n 8
rm -rf data/raw/img_train data/raw/img_val
zip -rq $Z data/yolo data/lists configs/streams.json
echo "data_prepared_$PROFILE.zip 저장 완료"

## 7. 변환 결과 눈으로 확인 (필수)
**확인할 것**: 초록 박스가 차·사람 위에 정확히 얹혀 있다. 어긋나면 **학습하지 말고** 멈춘다 (GUIDE 4.7절).

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/vis_*.jpg'))[:2]: display(Image(f))

## 8. Student 학습 (맑은 낮만)
세션이 끊기면 **같은 셀을 다시 실행**하면 `last.pt`에서 이어서 학습한다 (완료되면 `DONE` 표시 파일이 생겨 건너뛴다). **확인할 것**: 끝나면 `student 학습 완료`와 `mAP50` 값이 보인다. 이 값을 기록한다.

In [ ]:
%%bash
cd /content/edge-adapt
source profile.env
D=/content/drive/MyDrive/edge-adapt/runs/student
python scripts/make_yaml.py student > /dev/null
if [ -f $D/DONE_$PROFILE ]; then echo "이미 학습됨 ($PROFILE)"; else
  if [ -f $D/weights/last.pt ] && [ -f $D/STARTED_$PROFILE ]; then yolo detect train resume model=$D/weights/last.pt
  else mkdir -p $D && touch $D/STARTED_$PROFILE && yolo detect train data=configs/student.yaml model=yolo11n.pt imgsz=640 epochs=$STUDENT_EPOCHS batch=32 device=0 seed=0 project=/content/drive/MyDrive/edge-adapt/runs name=student exist_ok=True; fi
  touch $D/DONE_$PROFILE
fi
ls -la $D/weights; echo "student 학습 완료"

## 9. Teacher 학습 (맑음·비·밤, 안개 제외)
Teacher는 노트북 CPU 서버에서 **의사 라벨(추론)** 을 만들어야 하므로 `yolo11m`으로 했다. 끊기면 같은 셀을 다시 실행한다. **확인할 것**: `teacher 학습 완료`.

In [ ]:
%%bash
cd /content/edge-adapt
source profile.env
D=/content/drive/MyDrive/edge-adapt/runs/teacher
python scripts/make_yaml.py teacher > /dev/null
if [ -f $D/DONE_$PROFILE ]; then echo "이미 학습됨 ($PROFILE)"; else
  if [ -f $D/weights/last.pt ] && [ -f $D/STARTED_$PROFILE ]; then yolo detect train resume model=$D/weights/last.pt
  else mkdir -p $D && touch $D/STARTED_$PROFILE && yolo detect train data=configs/teacher.yaml model=yolo11m.pt imgsz=640 epochs=$TEACHER_EPOCHS batch=16 device=0 seed=0 project=/content/drive/MyDrive/edge-adapt/runs name=teacher exist_ok=True; fi
  touch $D/DONE_$PROFILE
fi
ls -la $D/weights; echo "teacher 학습 완료"

## 10. 기본 모델 내보내기와 어댑터 파이프라인 점검 (필수 관문)
**확인할 것**: `check_adapter_pipeline.py` 의 (1)~(3) 이 모두 통과다. 하나라도 실패하면 **10번 이후로 가지 말고** 출력 전체를 팀에 공유한다 (GUIDE 5.4절).

In [ ]:
%%bash
cd /content/edge-adapt
R=/content/drive/MyDrive/edge-adapt/runs
cp $R/teacher/weights/best.pt models/teacher.pt
python scripts/export_base.py $R/student/weights/best.pt
python scripts/check_adapter_pipeline.py

## 11. 환경별 성능 하락 측정 (Baseline A, Teacher 단독 T)
**확인할 것**: `env_map` 이 환경별 mAP 표를 출력한다 (clear 가 가장 높고 rain/night/fog 가 낮아야 한다). `env_sig_check` 의 `--env-thr` 권장값을 아래 셀 13의 `ENV_THR` 에 적는다.

In [ ]:
%%bash
cd /content/edge-adapt
python scripts/run_experiment.py --split detector_train --method A --tag A_dtrain
python scripts/run_experiment.py --split eval --method A --tag A
python scripts/run_experiment.py --split eval --method T --tag T
python scripts/env_map.py results/A_dtrain.json results/A.json results/T.json
python scripts/env_sig_check.py --split eval

## 12. 어댑터 설정 튜닝 (개발용 스트림에서만, 평가 스트림은 사용 안 함)
서버 적응의 학습 epoch·학습률·**헤드 앞에서 추가로 학습할 레이어 수**를 개발용 스트림의 보류 프레임 mAP로 고른다. 결과는 `configs/adapter.json`에 저장되고 이후 모든 실험(B, EdgeSync-lite, 제안)이 자동으로 같은 설정을 쓴다. **확인할 것**: 마지막 줄에 `선택: ...` 이 나오고 '평균 이득'이 양수다. `경고: 어느 설정도 ... 올리지 못했습니다`가 나오면 **본 실험으로 가지 말고** 출력 전체를 공유한다 (GUIDE 5.6절).

In [ ]:
%%bash
cd /content/edge-adapt
source profile.env
python scripts/tune_adapter.py $TUNE_FLAGS
cp configs/adapter.json /content/drive/MyDrive/edge-adapt/adapter.json

## 13. 감지기 학습 (RQ1) 과 파이프라인 시험
`DROP` 은 GUIDE 6.1절 규칙으로 정한다 (맑은 구간 mAP 표준편차의 2배 이상, 비·밤 평균 하락보다 작게). `ENV_THR` 는 셀 11 결과로 정한다.
**확인할 것**: 감지기 표의 Recall/오경보율, 그리고 `D_try` 의 `[server] {'sec': …}` (GPU 서버 기준 적응 1회 시간).

In [ ]:
DROP = 0.10      # <- 셀 11 결과로 바꾼다
ENV_THR = 0.08   # <- env_sig_check 권장값으로 바꾼다
import subprocess
print(subprocess.run(f'python scripts/train_detector.py --drop {DROP}', shell=True, capture_output=True, text=True).stdout[-3000:])
print(subprocess.run(f'python scripts/run_experiment.py --split eval --method D --selector diversity --budget 16 --seed 0 --delay 50 --env-thr {ENV_THR} --tag D_try', shell=True, capture_output=True, text=True))

## 14. 본 실험 (선별 4종 × 예산 × 시드, 기준선, EdgeSync-lite, Ablation)
규모는 `PROFILE` 이 정한다 (`free`: 예산 16, 시드 0 한 번 / `full`: 예산 8·16·32, 시드 3개). `DELAY` 는 **데모에 쓸 노트북 CPU 서버의 적응 소요 시간**으로 정하는 것이 가장 정직하다: 노트북에서 GUIDE 6.4절의 측정을 한 뒤 `ceil(sec×10)` 을 아래 셀의 `DELAY=100` 자리에 적는다. 측정 전이면 100 (=10초).
이미 끝난 실험은 자동으로 건너뛰므로 **끊기면 같은 셀을 다시 실행**한다. 목표 2(예산 3단계 중 2단계 이상)는 `full` 에서만 판정할 수 있다.

In [ ]:
%%bash
cd /content/edge-adapt
mkdir -p configs; [ -f /content/drive/MyDrive/edge-adapt/adapter.json ] && cp /content/drive/MyDrive/edge-adapt/adapter.json configs/adapter.json
source profile.env
export BUDGETS SEEDS DELAY=100
bash scripts/run_grid.sh

## 15. 분석과 목표 판정
**확인할 것**: `check_goals` 의 목표 2·3 판정과 **발표용 '제안 vs EdgeSync-lite'** 줄. 우세가 아니면 그대로 보고하고 GUIDE 6.7절의 점검 순서를 따른다.

In [ ]:
%%bash
cd /content/edge-adapt
python scripts/analyze.py
python scripts/check_goals.py

## 16. 노트북으로 가져갈 묶음 만들기
`laptop_bundle.zip` 이 드라이브에 생긴다. 드라이브 웹에서 다운로드해 노트북의 저장소 폴더에 풀면 된다 (GUIDE 8.1절). 학습된 `.pt` 와 평가 스트림, 감지기, 결과 표가 들어 있다. (ONNX 는 노트북에서 다시 내보낸다 — 토치 버전이 달라도 어댑터 이름이 어긋나지 않게 하기 위해서다.)

In [ ]:
%%bash
cd /content/edge-adapt
D=/content/drive/MyDrive/edge-adapt
R=$D/runs
cp $R/student/weights/best.pt models/student_best.pt
rm -f $D/laptop_bundle.zip
zip -rq $D/laptop_bundle.zip models/student_best.pt models/teacher.pt models/drift_logreg_*.pkl configs/streams.json data/yolo/stream_eval data/lists results/*.json results/summary.md results/curves.png 2>/dev/null || true
ls -la $D/laptop_bundle.zip